In [1]:
import os
import glob
import datetime

import numpy as np
import pandas as pd

import jax
import numpyro

import hssm
import arviz as az
from scipy.stats import gaussian_kde

import matplotlib.pyplot as plt
import seaborn as sns

import sqlite3

/Users/javierrojas/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
def get_fitted_parameters(df, participant_id):
    subset = df[df['participant_id'] == participant_id]
    v = subset[subset['param'] == 'v']['mean'].values[0]
    a = subset[subset['param'] == 'a']['mean'].values[0]
    z = subset[subset['param'] == 'z']['mean'].values[0]
    t = subset[subset['param'] == 't']['mean'].values[0]

In [3]:
def simulate_participant_ddm(participant_id, df, size=300):
    v, a, z, t = get_fitted_parameters(df, participant_id)
    v = np.repeat(v, size)          # drift rate
    a = a                           # boundary
    z = z                           # starting point
    t = t                           # non-decision time
    true_values = np.column_stack([v, np.repeat([[a, z, t]], size, axis=0)])

    dataset = hssm.simulate_data(
        model="ddm",
        theta=true_values,
        size=1,
    )

    dataset["participant_id"] = str(participant_id)
    return dataset

In [ ]:
def simulate_participant_ddm_th(participant_id, df, size=300):
    v, a, z, t = get_fitted_parameters(df, participant_id)
    v = np.repeat(v, size)          # drift rate
    a = a                           # boundary
    z = z                           # starting point
    t = t                           # non-decision time
    mod = 
    true_values = np.column_stack([v, np.repeat([[a, z, t]], size, axis=0)])

    dataset = hssm.simulate_data(
        model="ddm",
        theta=true_values,
        size=1,
    )

    dataset["participant_id"] = str(participant_id)
    return dataset

In [ ]:
def simulate_participant_ddm_from_dist(participant_id, size=300):
    """
    Simulate DDM data by sampling parameters from distributions.

    Parameters:
    - participant_id: identifier for the simulated participant
    - size: number of trials to simulate

    Parameter distributions (typical for DDM):
    - v (drift rate): Normal(1.5, 0.5) - typically 0.5 to 3.0
    - a (boundary separation): Normal(2.0, 0.3) - typically 1.0 to 3.0
    - z (starting point): Beta(2, 2) scaled to (0.2, 0.8) - typically 0.3 to 0.7
    - t (non-decision time): Normal(0.3, 0.1) - typically 0.1 to 0.5
    """

    # Sample parameters from distributions
    v_mean = np.random.normal(1.5, 0.5)  # drift rate
    a_val = np.random.normal(2.0, 0.3)   # boundary separation
    z_val = 0.2 + 0.6 * np.random.beta(2, 2)  # starting point (0.2 to 0.8)
    t_val = np.random.normal(0.3, 0.1)   # non-decision time

    # Ensure parameters are in reasonable ranges
    v_mean = np.clip(v_mean, 0.1, 4.0)
    a_val = np.clip(a_val, 0.5, 4.0)
    z_val = np.clip(z_val, 0.1, 0.9)
    t_val = np.clip(t_val, 0.05, 0.8)

    # Create parameter array for HSSM
    v = np.repeat(v_mean, size)
    a = np.repeat(a_val, size)
    z = np.repeat(z_val, size)
    t = np.repeat(t_val, size)

    true_values = np.column_stack([v, a, z, t])

    dataset = hssm.simulate_data(
        model="ddm",
        theta=true_values,
        size=1,
    )

    dataset["participant_id"] = str(participant_id)

    # Store the true parameters for later analysis
    dataset["true_v"] = v_mean
    dataset["true_a"] = a_val
    dataset["true_z"] = z_val
    dataset["true_t"] = t_val

    return dataset

In [4]:
def fit_hssm_participant(df, participant_column):
    all_summaries = []
    all_inferences = {}

    for nsub, isub in enumerate(df[participant_column].unique()):
        print(f"___Participant {isub}, {nsub+1}/{df[participant_column].nunique()}___")

        df_sub = df[df[participant_column] == isub].drop(columns=[participant_column])

        print("Median RT =", np.median(df_sub['rt']))
        print("N trials =", len(df_sub))

        model = hssm.HSSM(
            model="ddm",
            data=df_sub,
        )

        infer_data_sub = model.sample(
            cores=3,
            chains=3,
            draws=300,
            tune=1000,
            idata_kwargs=dict(log_likelihood=True),
            progressbar=True,
            target_accept=0.99,
        )

        all_inferences[isub] = infer_data_sub

        summary_df = (
            az.summary(infer_data_sub)
              .reset_index()
              .rename(columns={'index': 'param'})
        )
        summary_df['participant_id'] = isub
        all_summaries.append(summary_df)

    all_summaries_df = pd.concat(all_summaries, ignore_index=True)
    return all_summaries_df, all_inferences